In [23]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

ARROWS_DIR = Path("Images/arrows")

In [24]:
def load_grayscale(path):
    """Load a PNG and return a 2D array where each value is the mean of that pixel's R, G, B."""
    rgb = np.asarray(Image.open(path).convert("RGB"), dtype=np.float64)  # (H, W, 3), drops alpha
    return rgb.mean(axis=2)  # (H, W)

In [25]:
img = load_grayscale(ARROWS_DIR / "N_0.PNG")
print(img.shape, img.dtype, img.min(), img.max())
display(img)

(500, 500) float64 0.0 255.0


array([[255., 255., 255., ..., 255., 255., 255.],
       [255., 255., 255., ..., 255., 255., 255.],
       [255., 255., 255., ..., 255., 255., 255.],
       ...,
       [255., 255., 255., ..., 255., 255., 255.],
       [255., 255., 255., ..., 255., 255., 255.],
       [255., 255., 255., ..., 255., 255., 255.]], shape=(500, 500))

In [26]:
def append_to_csv(img, csv_path="arrows.csv", label=None):
    """Flatten a 2D array of pixel averages and append it as one row of csv_path.

    Columns are pixel0, pixel1, ... in row-major order (row 0 left-to-right, then row 1, ...).
    If label is given, it goes in a leading "label" column.
    The header is written only when the file doesn't exist yet.
    """
    csv_path = Path(csv_path)
    row = pd.DataFrame([img.ravel()], columns=[f"pixel{i}" for i in range(img.size)])
    if label is not None:
        row.insert(0, "label", label)
    row.to_csv(csv_path, mode="a", header=not csv_path.exists(), index=False)

In [27]:
CSV_PATH = Path("arrows.csv")
CSV_PATH.unlink(missing_ok=True)  # start fresh so re-running doesn't duplicate rows

png_paths = sorted(p for p in ARROWS_DIR.iterdir() if p.suffix.lower() == ".png")
for path in png_paths:
    label = path.stem.split("_")[0]  # "NE_0.PNG" -> "NE"
    append_to_csv(load_grayscale(path), CSV_PATH, label=label)
    print(f"{path.name} -> {label}")

print(f"Wrote {len(png_paths)} rows to {CSV_PATH}")

Background.PNG -> Background
E_0.PNG -> E
NE_0.PNG -> NE
NW_0.PNG -> NW
N_0.PNG -> N
SE_0.PNG -> SE
SW_0.PNG -> SW
S_0.PNG -> S
W_0.PNG -> W
Wrote 9 rows to arrows.csv
